In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score
from sklearn.utils import resample

In [7]:
summary = {'gender': [{'group': 'Male',
   'mean': 0.07,
   'se': 0.03,
   'ci_lower': 0.01,
   'ci_upper': 0.14,
   'n_total': 68,
   'n_wrong': 5},
  {'group': 'Female',
   'mean': 0.07,
   'se': 0.03,
   'ci_lower': 0.02,
   'ci_upper': 0.13,
   'n_total': 87,
   'n_wrong': 6},
  {'group': 'all',
   'mean': 0.07,
   'se': 0.02,
   'ci_lower': 0.03,
   'ci_upper': 0.11,
   'n_total': 155,
   'n_wrong': 11}],
 'age_group': [{'group': '60 - 79',
   'mean': 0.02,
   'se': 0.02,
   'ci_lower': -0.01,
   'ci_upper': 0.06,
   'n_total': 80,
   'n_wrong': 2},
  {'group': '20 - 39',
   'mean': 0.14,
   'se': 0.07,
   'ci_lower': -0.01,
   'ci_upper': 0.28,
   'n_total': 21,
   'n_wrong': 3},
  {'group': '40 - 59',
   'mean': 0.09,
   'se': 0.04,
   'ci_lower': 0.0,
   'ci_upper': 0.17,
   'n_total': 44,
   'n_wrong': 4},
  {'group': 'Unknown',
   'mean': 0.2,
   'se': 0.13,
   'ci_lower': -0.05,
   'ci_upper': 0.45,
   'n_total': 10,
   'n_wrong': 2},
  {'group': 'all',
   'mean': 0.07,
   'se': 0.02,
   'ci_lower': 0.03,
   'ci_upper': 0.11,
   'n_total': 155,
   'n_wrong': 11}],
 'race': [{'group': 'White',
   'mean': 0.07,
   'se': 0.02,
   'ci_lower': 0.03,
   'ci_upper': 0.12,
   'n_total': 135,
   'n_wrong': 10},
  {'group': 'Non-White',
   'mean': 0.05,
   'se': 0.05,
   'ci_lower': -0.04,
   'ci_upper': 0.14,
   'n_total': 20,
   'n_wrong': 1},
  {'group': 'all',
   'mean': 0.07,
   'se': 0.02,
   'ci_lower': 0.03,
   'ci_upper': 0.11,
   'n_total': 155,
   'n_wrong': 11}]}

In [8]:
from statsmodels.stats.proportion import proportions_ztest


# Extract gender data for male and female only
gender_data = summary['gender']
female = next(g for g in gender_data if g['group'] == 'Female')
male = next(g for g in gender_data if g['group'] == 'Male')

# Gather counts
counts = [female['n_wrong'], male['n_wrong']]
totals = [female['n_total'], male['n_total']]
props = [counts[i] / totals[i] for i in range(2)]

# Check if sample size conditions are met for z-test
conditions_met = all([
    totals[i] * props[i] >= 5 and totals[i] * (1 - props[i]) >= 5
    for i in range(2)
])

# Perform test if valid
if conditions_met:
    stat, pval = proportions_ztest(counts, totals)
    gender_result = {
        "test": "z-test for two proportions",
        "z_statistic": round(stat, 4),
        "p_value": round(pval, 4),
        "female_error_rate": round(props[0], 3),
        "male_error_rate": round(props[1], 3),
        "sample_size_conditions_met": True
    }
else:
    result = {
        "error": "Sample size requirements not met for z-test",
        "sample_size_conditions_met": False
    }

gender_result

{'test': 'z-test for two proportions',
 'z_statistic': -0.1098,
 'p_value': 0.9126,
 'female_error_rate': 0.069,
 'male_error_rate': 0.074,
 'sample_size_conditions_met': True}

In [9]:
# Extract race data for White and Non-White
race_data = summary['race']
white = next(r for r in race_data if r['group'].lower() == 'white')
non_white = next(r for r in race_data if r['group'].lower() == 'non-white')

# Gather counts
counts = [white['n_wrong'], non_white['n_wrong']]
totals = [white['n_total'], non_white['n_total']]
props = [counts[i] / totals[i] for i in range(2)]

# Check sample size conditions
conditions_met_race = all([
    totals[i] * props[i] >= 5 and totals[i] * (1 - props[i]) >= 5
    for i in range(2)
])

# Perform test if valid
if conditions_met_race:
    stat, pval = proportions_ztest(counts, totals)
    race_result = {
        "test": "z-test for two proportions",
        "z_statistic": round(stat, 4),
        "p_value": round(pval, 4),
        "white_error_rate": round(props[0], 3),
        "non_white_error_rate": round(props[1], 3),
        "sample_size_conditions_met": True
    }
else:
    race_result = {
        "error": "Sample size requirements not met for z-test",
        "sample_size_conditions_met": False
    }

race_result

{'error': 'Sample size requirements not met for z-test',
 'sample_size_conditions_met': False}

In [10]:
from scipy.stats import fisher_exact

# 1. Set up the 2x2 contingency table
# Format: 
# [ [Group 1 Errors, Group 1 Correct],
#   [Group 2 Errors, Group 2 Correct] ]

white_wrong = counts[0]
white_correct = totals[0] - counts[0]

non_white_wrong = counts[1]
non_white_correct = totals[1] - counts[1]

contingency_table = [
    [white_wrong, white_correct],        # White: [10, 128]
    [non_white_wrong, non_white_correct] # Non-White: [1, 21]
]

# 2. Perform Fisher's Exact Test
odds_ratio, p_value = fisher_exact(contingency_table)

# 3. Format the output
race_result = {
    "test": "Fisher's Exact Test",
    "odds_ratio": round(odds_ratio, 4),
    "p_value": round(p_value, 4),
    "white_error_rate": round(props[0], 3),
    "non_white_error_rate": round(props[1], 3),
    "sample_size_conditions_met": "Not required for Fisher's Exact"
}

race_result

{'test': "Fisher's Exact Test",
 'odds_ratio': 1.52,
 'p_value': 1.0,
 'white_error_rate': 0.074,
 'non_white_error_rate': 0.05,
 'sample_size_conditions_met': "Not required for Fisher's Exact"}

In [11]:
from scipy.stats import chi2_contingency

# Extract age group data and filter valid ones
age_data = summary['age_group']
valid_groups = [g for g in age_data if g['group'] in ['20 - 39', '40 - 59', '60 - 79']]

# Create contingency table: [correct, incorrect] for each group
age_contingency = []
age_labels = []

for group in valid_groups:
    correct = group['n_total'] - group['n_wrong']
    incorrect = group['n_wrong']
    age_contingency.append([correct, incorrect])
    age_labels.append(group['group'])

# Run chi-square test of independence
chi2_stat, p_val, dof, expected = chi2_contingency(age_contingency)

# Create expected frequencies DataFrame and check for < 5
expected_df = pd.DataFrame(expected, columns=['Correct_exp', 'Incorrect_exp'], index=age_labels)
expected_check = (expected_df < 5).any(axis=1)
any_cell_under_5 = expected_check.any()

# Format result
age_result = {
    "test": "Chi-square test of independence",
    "chi2_statistic": round(chi2_stat, 4),
    "p_value": round(p_val, 4),
    "degrees_of_freedom": dof,
    "age_groups_compared": age_labels,
    "all_expected_freqs_>=5": not any_cell_under_5
}

age_result


{'test': 'Chi-square test of independence',
 'chi2_statistic': 4.8713,
 'p_value': 0.0875,
 'degrees_of_freedom': 2,
 'age_groups_compared': ['60 - 79', '20 - 39', '40 - 59'],
 'all_expected_freqs_>=5': False}

In [12]:
from statsmodels.stats.multitest import multipletests

# Raw p-values from your tests
pvals = [gender_result['p_value'], race_result['p_value'], age_result['p_value']]
labels = ['Gender', 'Race', 'Age Group']

# Apply FDR correction (Benjamini-Hochberg)
reject, pvals_corrected, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

# Display results
for i in range(len(pvals)):
    print(f"{labels[i]}: raw p = {pvals[i]:.4f}, FDR-corrected p = {pvals_corrected[i]:.4f}, significant = {reject[i]}")


Gender: raw p = 0.9126, FDR-corrected p = 1.0000, significant = False
Race: raw p = 1.0000, FDR-corrected p = 1.0000, significant = False
Age Group: raw p = 0.0875, FDR-corrected p = 0.2625, significant = False
